In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys

# sys.path.insert(0, "/home/matis/code/Florian-Q/maraicherbio-prediction/notebooks/")

In [4]:
import utils
import utils_series
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
# --- Boucle : df_train / df_test pour chaque produit (split adaptatif, date fin commune) ---
df_all = utils.charger_dataframe()

# Définir la date de fin commune pour TOUS les splits
utils_series.GLOBAL_TEST_END_DATE = df_all['created'].max()
print(f'Date de fin commune : {utils_series.GLOBAL_TEST_END_DATE.date()}\n')

modeles = sorted(df_all['model'].unique())
print(f'{len(modeles)} produits à traiter\n')

dict_train = {}
dict_test = {}
skipped = []

for modele in modeles:
    try:
        df_produit = utils.charger_dataframe(modele)
        df_model = utils_series.complete_weekly_dataframe(df_produit, 'created', 'quantite_y')
        train, test = utils_series.split_adaptive_seasonal(df_model, test_pct=0.20)
        dict_train[modele] = train
        dict_test[modele] = test
    except ValueError as e:
        skipped.append(modele)
        continue

print(f'\nTerminé : {len(dict_train)} produits prêts  |  {len(skipped)} ignorés')
if skipped:
    print(f'Ignorés : {skipped}')

# Vérification
test_ends = [t.index.max().date() for t in dict_test.values()]
print(f'Toutes les fins de test = {test_ends[0]} ?  {len(set(test_ends)) == 1}')

Date de fin commune : 2026-05-27

100 produits à traiter

Train : 2014-06-08  →  2024-05-26  (521 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 12.0 ans (17%)
Train : 2014-07-06  →  2024-05-26  (517 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2014-06-29  →  2024-05-26  (518 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2015-01-11  →  2024-05-26  (490 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.4 ans (18%)
Train : 2018-05-06  →  2024-05-26  (317 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 8.1 ans (25%)
Train : 2014-01-05  →  2024-05-26  (543 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 12.4 ans (16%)
Train : 2014-02-23  →  2024-05-26  (536 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an

In [6]:
# ============================================================
# PROPHET v1 : Moyenne hebdo + split_adaptive + seasonal_metrics
# ============================================================
import numpy as np
import pandas as pd
import itertools
from prophet import Prophet

# ── Hyperparamètres fixes ─────────────────────────────────────────────────────
BEST_PARAMS: dict = {
    "growth"              : "flat",       # pas de tendance visible
    "seasonality_mode"    : "additive",
    "weekly_seasonality"  : True,
    "daily_seasonality"   : False,
    "yearly_seasonality"  : True,
    "interval_width"      : 0.95,
    "fourier_order"       : 0,            # inutile ici → désactivé
    "holidays_prior_scale": 1,            # absorbe les pics ponctuels
}

# ── Grid Search ───────────────────────────────────────────────────────────────
TUNING_GRID: dict = {
    "changepoint_prior_scale" : [0.2, 0.3, 0.4, 0.5],
    "seasonality_prior_scale" : [3, 5, 7, 10],
}

ALL_COMBOS = list(itertools.product(
    TUNING_GRID["changepoint_prior_scale"],
    TUNING_GRID["seasonality_prior_scale"],
))
# 4 × 4 = 16 combinaisons par produit

# ── Helpers ───────────────────────────────────────────────────────────────────

def _build_prophet_df(series: pd.Series) -> pd.DataFrame:
    """Convertit une Series indexée en datetime → DataFrame Prophet (ds, y)."""
    return (
        series
        .reset_index()
        .rename(columns={series.index.name or "index": "ds", series.name or 0: "y"})
        [["ds", "y"]]
    )


def _fit_predict_prophet(y_train: pd.Series, y_test: pd.Series,
                         cps: float, sps: float) -> np.ndarray:
    """
    Entraîne Prophet sur y_train, prédit sur les dates de y_test.
    Retourne un tableau numpy clipé à 0.
    """
    df_train = _build_prophet_df(y_train)

    m = Prophet(
        growth                 = BEST_PARAMS["growth"],
        seasonality_mode       = BEST_PARAMS["seasonality_mode"],
        weekly_seasonality     = BEST_PARAMS["weekly_seasonality"],
        daily_seasonality      = BEST_PARAMS["daily_seasonality"],
        yearly_seasonality     = BEST_PARAMS["yearly_seasonality"],
        interval_width         = BEST_PARAMS["interval_width"],
        changepoint_prior_scale= cps,
        seasonality_prior_scale= sps,
        holidays_prior_scale   = BEST_PARAMS["holidays_prior_scale"],
    )
    # fourier_order = 0 → on n'ajoute aucune saisonnalité custom supplémentaire
    m.fit(df_train)

    future = pd.DataFrame({"ds": y_test.index})
    forecast = m.predict(future)
    y_pred = np.clip(forecast["yhat"].values, 0, None)
    return y_pred


# ── Boucle principale ─────────────────────────────────────────────────────────
results = []

for modele in dict_train.keys():
    y_train = dict_train[modele]['quantite_y']
    y_test  = dict_test[modele]['quantite_y']

    best_smape = np.inf
    best_pred  = None
    best_cps   = None
    best_sps   = None

    # — Grid-search sur le jeu de test (choix du meilleur combo) —
    for cps, sps in ALL_COMBOS:
        try:
            y_pred = _fit_predict_prophet(y_train, y_test, cps, sps)
            m_tmp  = utils_series.seasonal_metrics(y_test, y_pred)
            if m_tmp["MAE_in"] < best_smape:
                best_smape = m_tmp["MAE_in"]
                best_pred  = y_pred
                best_cps   = cps
                best_sps   = sps
        except Exception as e:
            print(f"[WARN] {modele} | cps={cps} sps={sps} → {e}")
            continue

    if best_pred is None:          # tous les combos ont échoué
        continue

    metrics = utils_series.seasonal_metrics(y_test, best_pred)

    results.append({
        'produit'    : modele,
        'train_debut': y_train.index.min().strftime('%Y-%m'),
        'test_fin'   : y_test.index.max().strftime('%Y-%m'),
        'train_sem'  : len(y_train),
        'test_sem'   : len(y_test),
        'best_cps'   : best_cps,
        'best_sps'   : best_sps,
        'pct_zeros'  : round(metrics['pct_zeros'], 1),
        'MAE_in'     : round(metrics['MAE_in'],    2),
        'MAE_out'    : round(metrics['MAE_out'],   2),
        'MAE_all'    : round(metrics['MAE_all'],   2),
        'MAPE'       : round(metrics['MAPE'],      1),
        'sMAPE_all'  : round(metrics['sMAPE_all'], 1),
    })

# ── Tableau ───────────────────────────────────────────────────────────────────
df_prophet = pd.DataFrame(results).sort_values('sMAPE_all')
print(f'PROPHET v1 — {len(df_prophet)} produits '
      f'(date fin commune : {df_prophet["test_fin"].iloc[0]})\n')
print(df_prophet.to_string(index=False))

# ── Résumé ────────────────────────────────────────────────────────────────────
print(f'\nMAE_all moyen : {df_prophet["MAE_all"].mean():.2f}  |  '
      f'MAE_in moyen : {df_prophet["MAE_in"].mean():.2f}  |  '
      f'MAE_out moyen : {df_prophet["MAE_out"].mean():.2f}')
print(f'sMAPE médian  : {df_prophet["sMAPE_all"].median():.1f} %  |  '
      f'Zéros moyen  : {df_prophet["pct_zeros"].mean():.1f} %')

# ── Top 5 / Bottom 5 ──────────────────────────────────────────────────────────
print(f'\n--- Top 5 (sMAPE) ---')
print(df_prophet[['produit', 'MAE_all', 'MAPE', 'sMAPE_all',
                  'best_cps', 'best_sps']].head(5).to_string(index=False))
print(f'\n--- Bottom 5 (sMAPE) ---')
print(df_prophet[['produit', 'MAE_all', 'MAPE', 'sMAPE_all',
                  'best_cps', 'best_sps']].tail(5).to_string(index=False))

/home/flo/.pyenv/versions/maraicherbio-prediction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
16:14:24 - cmdstanpy - INFO - Chain [1] start processing
16:14:24 - cmdstanpy - INFO - Chain [1] done processing
16:14:24 - cmdstanpy - INFO - Chain [1] start processing
16:14:24 - cmdstanpy - INFO - Chain [1] done processing
16:14:24 - cmdstanpy - INFO - Chain [1] start processing
16:14:24 - cmdstanpy - INFO - Chain [1] done processing
16:14:24 - cmdstanpy - INFO - Chain [1] start processing
16:14:24 - cmdstanpy - INFO - Chain [1] done processing
16:14:24 - cmdstanpy - INFO - Chain [1] start processing
16:14:24 - cmdstanpy - INFO - Chain [1] done processing
16:14:24 - cmdstanpy - INFO - Chain [1] start processing
16:14:24 - cmdstanpy - INFO - Chain 

PROPHET v1 — 84 produits (date fin commune : 2026-05)

                      produit train_debut test_fin  train_sem  test_sem  best_cps  best_sps  pct_zeros  MAE_in  MAE_out  MAE_all  MAPE  sMAPE_all
              Betterave cuite     2014-01  2026-05        543       105       0.2         3        3.8    2.32     2.15     2.31  52.6       73.6
       Pain multigraines 500g     2019-12  2026-05        286        53       0.2         3       17.0    1.21     2.01     1.34  61.0       74.7
                    Courgette     2014-05  2026-05        525       105       0.2        10       50.5    5.32     0.13     2.70  37.9       79.1
                      Poireau     2014-01  2026-05        543       104       0.2         3       51.0    3.92     1.12     2.50  47.7       80.7
              Aubergine noire     2014-07  2026-05        517       104       0.2        10       68.3    3.38     0.56     1.45  41.2       83.7
              tomate ancienne     2019-08  2026-05        304        

In [13]:
df_prophet['MAPE'].mean()

np.float64(74.18809523809523)

In [8]:
df_prophet.sort_values("MAPE", ascending=False).head(20)

,produit,train_debut,test_fin,train_sem,test_sem,best_cps,best_sps,pct_zeros,MAE_in,MAE_out,MAE_all,MAPE,sMAPE_all
38,Navet,2014-01,2026-05,543,104,0.2,3,70.2,1.11,0.49,0.68,228.7,129.2
33,Jeune pousse epinard,2022-04,2026-05,162,52,0.2,3,84.6,0.67,0.05,0.15,156.5,125.9
72,Topinambour,2014-01,2026-05,543,104,0.2,3,71.2,1.01,0.35,0.54,147.9,117.2
8,CHOU FLEUR,2023-01,2026-05,125,52,0.2,3,75.0,9.82,1.21,3.36,131.8,127.8
4,Betterave botte,2018-05,2026-05,317,105,0.2,3,64.8,2.50,1.28,1.71,130.4,134.2
40,Navet boule d'or,2015-01,2026-05,489,104,0.2,3,66.3,0.98,0.72,0.81,128.6,121.4
15,Choux pointu,2014-05,2026-05,523,104,0.2,3,88.5,2.56,0.33,0.59,126.4,131.0
68,Salade Feuille de chêne verte,2014-04,2026-05,528,105,0.2,3,45.7,2.77,2.70,2.74,109.3,121.6
2,Basilic,2014-06,2026-05,518,105,0.2,3,63.8,3.38,0.06,1.26,107.7,103.1
39,Navet botte,2014-01,2026-05,542,105,0.2,3,56.2,3.40,1.11,2.11,101.4,127.8


In [9]:
# Ajouter la colonne weight_units (sans écraser df_base)
unit_map = df_all[['model', 'weight_units']].drop_duplicates().set_index('model')['weight_units']
df_base_units = df_base.copy()
df_base_units['weight_units'] = df_base_units['produit'].map(unit_map)
df_base_units

NameError: name 'df_base' is not defined

In [ ]:
df_prophet['sMAPE_all']

In [ ]:
df_base_units.to_csv('../data/Baseline_metrics.csv', index=False)